# Synthetic sensor faults — validation/test CSV benchmark generator

**Purpose:** Generate controlled sensor-level fault datasets as CSV files to evaluate an anomaly detector (e.g., Isolation Forest). This notebook **does not train the model**.

### Dataset roles

| Split | Role | Synthetic fault injection? |
|---|---|---|
| `train.csv` | Train the anomaly detector independently; estimate sensor mean/SD used to scale injected faults | **No** |
| `val.csv` | Development benchmarks: compare fault detection and choose reading windows/thresholds | **Yes — default** |
| `test.csv` | Final benchmarks after model selection and threshold calibration are frozen | **Yes — only when selected** |

Use **`BENCHMARK_SPLIT = "validation"`** for routine benchmark generation. Change it to `"test"` only for final held-out evaluation. Each split uses its **own output directory** so test benchmark files do not overwrite validation results.

**About `PW_0.5h`:** `1` means a machine failure will occur within the next **30 minutes**; it does **not** mean the machine is currently in a failed state. This generator conservatively injects faults only in periods labeled `PW_0.5h = 0` (including the preceding reading-window lookback). A label of `0` does **not** guarantee a completely healthy sensor or machine.

**Separate objectives:** `synthetic_anomaly` identifies deliberately injected *sensor faults*. It is distinct from the original machine **future-failure** target `PW_0.5h`. Neither label should be included among an unsupervised detector's input sensor features.

Each experiment produces one new, full-length CSV containing a single injected event. Input CSV files remain unchanged. A `manifest.csv` records the source dataset, injection details, and output paths. Avoid concatenating experiment copies into one training set, since most readings would be repeated.

**Requirements:** Python, `numpy`, `pandas`, and Jupyter/IPython. Execute cells top to bottom.

## 1. Configure datasets and experiment size

In [1]:
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
from IPython.display import display

# Data source roles: always use TRAIN_PATH for reference mean/SD.
DATA_FOLDER = Path("PW_0.5h")
TRAIN_PATH = DATA_FOLDER / "train.csv"
VALIDATION_PATH = DATA_FOLDER / "val.csv"  # change to "validation.csv" if that is the filename
TEST_PATH = DATA_FOLDER / "test.csv"
BENCHMARK_SPLIT = "validation"               # "validation" or "test" (never "train")

# Different output folders prevent validation and final-test outputs from overwriting each other.
assert BENCHMARK_SPLIT in ("validation", "test")
OUTPUT_DIR = Path(f"synthetic_{BENCHMARK_SPLIT}_fault_outputs")

MODE = "pilot"                    # "pilot" or "full"; full creates 12,300 CSVs using the defaults
RANDOM_SEED = 42
PILOT_REPLICATES = 2
FULL_REPLICATES = 20
FULL_SEVERITIES_SD = (0.5, 1.0, 2.0, 3.0, 5.0)
FULL_DURATIONS_SAMPLES = (1, 3, 5, 10, 30, 60)
BENIGN_NOISE_SEVERITIES_SD = (0.02, 0.05, 0.10)
PILOT_SEVERITY_SD = 2.0
PILOT_DURATION_SAMPLES = 5
READING_WINDOWS = (5, 10, 30, 60)  # used to require eligible history, not to train models

TIMESTAMP_COL = "timestamp"
FAILURE_HORIZON_COL = "PW_0.5h"
BINARY_FEATURE_COL = "Oxygen over threshold"
SENSOR_COLS = (
    "CMS air pressure",
    "Concentrazione ossigeno base",
    "Nitrogen pressure",
)
FAULT_TYPES = (
    "gaussian_noise", "spike", "short_burst", "random_corruption", "dropout",
    "bias", "drift", "stuck_sensor", "increased_noise", "intermittent_fault",
)

assert MODE in ("pilot", "full")
assert PILOT_REPLICATES > 0 and FULL_REPLICATES > 0
assert all(w >= 1 for w in READING_WINDOWS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "benchmarks").mkdir(parents=True, exist_ok=True)


## 2. Load the training reference and selected benchmark source

`train.csv` is loaded **only** to calculate the reference mean and standard deviation for each sensor. The dataset receiving injected faults is **`val.csv`** by default; it becomes **`test.csv`** if `BENCHMARK_SPLIT` is changed to `"test"`. The original files are not modified.

In [2]:
def load_and_validate(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing input file: {path.resolve()}")
    df = pd.read_csv(path)
    expected = [TIMESTAMP_COL, *SENSOR_COLS, BINARY_FEATURE_COL]
    missing = [col for col in expected if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing required columns {missing}")
    if df.columns[-1] != FAILURE_HORIZON_COL:
        raise ValueError(f"The final column must be {FAILURE_HORIZON_COL!r}.")
    if FAILURE_HORIZON_COL not in df:
        raise ValueError("The dataset needs a binary 0/1 future-failure label.")
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="raise")
    if df[TIMESTAMP_COL].isna().any() or not df[TIMESTAMP_COL].is_monotonic_increasing:
        raise ValueError(f"{path.name}: timestamps must be valid and increasing.")
    if df[TIMESTAMP_COL].duplicated().any():
        raise ValueError(f"{path.name}: duplicate timestamps detected.")
    for col in SENSOR_COLS:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col].to_numpy(dtype=float)).all():
            raise ValueError(f"{path.name}: sensor {col} has missing or infinite values.")
    if FAILURE_HORIZON_COL in df:
        if not df[FAILURE_HORIZON_COL].isin([0, 1]).all():
            raise ValueError(f"{path.name}: future-failure label must be binary 0/1.")
        df[FAILURE_HORIZON_COL] = df[FAILURE_HORIZON_COL].astype('int8')
    return df.reset_index(drop=True)


SOURCE_PATHS = {"validation": VALIDATION_PATH, "test": TEST_PATH}
BENCHMARK_PATH = SOURCE_PATHS[BENCHMARK_SPLIT]

clean_train = load_and_validate(TRAIN_PATH)
clean_benchmark = load_and_validate(BENCHMARK_PATH)

print(f"Reference statistics from: {TRAIN_PATH} | {len(clean_train):,} rows")
print(f"Injecting into {BENCHMARK_SPLIT}: {BENCHMARK_PATH} | {len(clean_benchmark):,} rows")
print(f"Eligible-label candidates (PW_0.5h=0) in source: "
      f"{(clean_benchmark[FAILURE_HORIZON_COL] == 0).sum():,} readings")

Reference statistics from: PW_0.5h\train.csv | 40,354 rows
Injecting into validation: PW_0.5h\val.csv | 34,344 rows
Eligible-label candidates (PW_0.5h=0) in source: 33,804 readings


## 3. Calculate training-based sensor statistics and valid injection positions

Severity in standard-deviation (SD) units is based **only on `train.csv` rows with `PW_0.5h = 0`**. This avoids estimating fault strength from validation/test sensor distributions. Those rows are plausible baseline candidates, not guaranteed healthy data.

Event placement is evaluated on **`clean_benchmark`** (validation or test), not `clean_train`: candidate intervals and their preceding `max(READING_WINDOWS)-1` readings must have future-failure label 0 and contiguous one-minute timestamps. This is a conservative experimental restriction to keep injected sensor faults separate from the labeled pre-failure horizon.

In [3]:
reference_source = "train (PW_0.5h=0 baseline candidates)"
reference_df = clean_train.loc[clean_train[FAILURE_HORIZON_COL].eq(0)]  # excludes labeled imminent-failure periods
if len(reference_df) < 3:
    raise ValueError("Insufficient PW_0.5h=0 training readings to estimate sensor standard deviations.")
REFERENCE_STATS = {
    col: {
        "mean": float(reference_df[col].mean()),
        "std": float(reference_df[col].std(ddof=1)),
        "min": float(reference_df[col].min()),
        "max": float(reference_df[col].max()),
    }
    for col in SENSOR_COLS
}
if any(not np.isfinite(s["std"]) or s["std"] <= 0 for s in REFERENCE_STATS.values()):
    raise ValueError("Some PW_0.5h=0 baseline sensors have zero/invalid SD. Adjust baseline data.")
print("Reference source:", reference_source)
display(pd.DataFrame(REFERENCE_STATS).T)

outside_failure_horizon = clean_benchmark[FAILURE_HORIZON_COL].to_numpy() == 0
steps = clean_benchmark[TIMESTAMP_COL].diff().dt.total_seconds().to_numpy()
regular_step = np.r_[True, np.isclose(steps[1:], 60.0)]
# Exclude readings with positive pre-failure labels and positions after timestamp gaps.
bad = ~outside_failure_horizon | ~regular_step
bad_prefix = np.r_[0, np.cumsum(bad.astype(np.int64))]
lookback = max(READING_WINDOWS) - 1

def eligible_starts(duration):
    """Starts with PW_0.5h=0 throughout contiguous lookback and injection interval."""
    duration = int(duration)
    if duration <= 0 or duration > len(clean_benchmark):
        return np.array([], dtype=np.int64)
    starts = np.arange(lookback, len(clean_benchmark) - duration + 1, dtype=np.int64)
    valid = bad_prefix[starts + duration] == bad_prefix[starts - lookback]
    return starts[valid]

eligible_cache = {d: eligible_starts(d) for d in FULL_DURATIONS_SAMPLES if d <= len(clean_benchmark)}
print("Eligible starts (no positive failure-horizon labels in lookback):", {k:len(v) for k,v in eligible_cache.items()})

Reference source: train (PW_0.5h=0 baseline candidates)


,mean,std,min,max
CMS air pressure,0.009538,1.001844,-0.948800,1.393893
Concentrazione ossigeno base,-0.002112,1.005186,-1.387926,11.447540
Nitrogen pressure,-0.009531,1.007078,-1.394478,1.481007


Eligible starts (no positive failure-horizon labels in lookback): {1: 32683, 3: 32645, 5: 32607, 10: 32512, 30: 32132, 60: 31562}


## 4. Configure the ten synthetic fault types

The existing `inject_fault()` function is preserved. Each output is a copy of the chosen benchmark source with **one** sensor altered for **one** event. `gaussian_noise` is deliberately labeled a benign negative control (`synthetic_anomaly = 0`); `intermittent_fault` labels only active corrupted phases. `dropout` introduces missing sensor values (`NaN`).

`MODE = "pilot"` generates 60 experiments with the defaults. `MODE = "full"` generates 12,300 experiments and stores one **full-length** dataset per experiment, requiring substantial disk space.

The below table shows the mechanism for each fault:
| Fault | Injection mechanism | `synthetic_anomaly` label |
|---|---|---|
| `gaussian_noise` | Low-amplitude Gaussian jitter, SD × 0.02–0.10 | **0** — benign *negative control* (not a labeled fault) |
| `spike` | ± severity × SD on a single reading | One reading = 1 |
| `short_burst` | Consecutive spikes with constant signed offset | Every active reading = 1 |
| `random_corruption` | Replace readings by randomly drawn outlying values around reference mean | Every active reading = 1 |
| `dropout` | Replace readings by `NaN` | Every active reading = 1 |
| `bias` | Persistent ± severity × SD offset | Every active reading = 1 |
| `drift` | Gradually increasing signed offset | Every active reading = 1 |
| `stuck_sensor` | Hold previous reading's value (may coincide with some unmodified values) | Entire stuck interval = 1 |
| `increased_noise` | Add higher-variance Gaussian noise | Every active reading = 1 |
| `intermittent_fault` | Alternate between faulty and untouched readings | **Only actually faulty phases** = 1 |

In [4]:
def experiment_specs(mode):
    durations = (PILOT_DURATION_SAMPLES,) if mode == "pilot" else FULL_DURATIONS_SAMPLES
    replicates = PILOT_REPLICATES if mode == "pilot" else FULL_REPLICATES
    for sensor in SENSOR_COLS:
        for fault_type in FAULT_TYPES:
            if fault_type == "gaussian_noise":
                severities = (0.05,) if mode == "pilot" else BENIGN_NOISE_SEVERITIES_SD
            elif fault_type in ("dropout", "stuck_sensor"):
                severities = (None,)
            else:
                severities = (PILOT_SEVERITY_SD,) if mode == "pilot" else FULL_SEVERITIES_SD

            valid_durations = (1,) if fault_type == "spike" else durations
            if fault_type == "short_burst":
                valid_durations = tuple(d for d in valid_durations if d >= 3)
            if fault_type == "intermittent_fault":
                valid_durations = tuple(d for d in valid_durations if d >= 2)
            for severity, duration, replicate in product(severities, valid_durations, range(1, replicates + 1)):
                yield {"sensor":sensor, "fault_type":fault_type,
                       "severity_sd":severity, "duration":int(duration), "replicate":replicate}


specifications = list(experiment_specs(MODE))
print(f"Planned CSV files: {len(specifications):,}")
if MODE == "full":
    print("CAUTION: full mode writes 12,300 whole-dataset CSV files; verify disk space first.")

Planned CSV files: 60


In [5]:
def inject_fault(clean_df, sensor, fault_type, start_index, duration, severity_sd, seed, event_id):
    """Inject exactly one synthetic event and return a new frame plus its event metadata."""
    if sensor not in SENSOR_COLS or fault_type not in FAULT_TYPES:
        raise ValueError("Unexpected sensor or fault type")
    start_index, duration = int(start_index), int(duration)
    end_exclusive = start_index + duration
    if duration < 1 or start_index < 1 or end_exclusive > len(clean_df):
        raise ValueError("Invalid fault interval")
    if not clean_df[FAILURE_HORIZON_COL].iloc[start_index:end_exclusive].eq(0).all():
        raise ValueError("Cannot inject during a PW_0.5h=1 pre-failure interval")
    if fault_type in ("dropout", "stuck_sensor"):
        severity_sd = None
    elif severity_sd is None or not np.isfinite(float(severity_sd)) or float(severity_sd) <= 0:
        raise ValueError("Positive severity required")

    rng = np.random.default_rng(int(seed))
    out = clean_df.copy(deep=True)
    values = out[sensor].to_numpy(dtype=float).copy()
    baseline = values[start_index:end_exclusive].copy()
    sd = REFERENCE_STATS[sensor]["std"]
    signed_direction = int(rng.choice([-1, 1]))
    amplitude = float(severity_sd) * sd if severity_sd is not None else None
    positive_phase = np.ones(duration, dtype=bool)

    if fault_type == "gaussian_noise":
        values[start_index:end_exclusive] += rng.normal(0, amplitude, duration)
        positive_phase[:] = False  # intended harmless measurement jitter
    elif fault_type == "spike":
        if duration != 1:
            raise ValueError("Spike must have duration 1")
        values[start_index] += signed_direction * amplitude
    elif fault_type == "short_burst":
        if duration < 3:
            raise ValueError("Short burst needs at least 3 readings")
        values[start_index:end_exclusive] += signed_direction * amplitude
    elif fault_type == "random_corruption":
        # Replace rather than add. Reference mean ± (1..2) × severity × SD.
        offsets = rng.uniform(1.0, 2.0, duration) * amplitude
        signs = rng.choice([-1.0, 1.0], size=duration)
        values[start_index:end_exclusive] = REFERENCE_STATS[sensor]["mean"] + signs * offsets
    elif fault_type == "dropout":
        values[start_index:end_exclusive] = np.nan
    elif fault_type == "bias":
        values[start_index:end_exclusive] += signed_direction * amplitude
    elif fault_type == "drift":
        # Ramp from just above zero to full severity to avoid a labeled-but-unmodified first step.
        ramp = np.linspace(0.0, 1.0, duration + 1)[1:]
        values[start_index:end_exclusive] += signed_direction * amplitude * ramp
    elif fault_type == "stuck_sensor":
        values[start_index:end_exclusive] = values[start_index - 1]
    elif fault_type == "increased_noise":
        values[start_index:end_exclusive] += rng.normal(0, amplitude, duration)
    elif fault_type == "intermittent_fault":
        positive_phase = (np.arange(duration) % 2 == 0)
        slc = values[start_index:end_exclusive].copy()
        slc[positive_phase] += signed_direction * amplitude
        values[start_index:end_exclusive] = slc

    # For some stuck-sensor placements, the held value equals normal readings in part
    # or all of the interval. Reject wholly invisible events (but not partial equality).
    changed_within = ~np.isclose(values[start_index:end_exclusive], baseline,
                                 atol=1e-12, rtol=0.0, equal_nan=True)
    if fault_type != "gaussian_noise" and not changed_within.any():
        raise ValueError("Invisible fault (no observable sensor change)")

    out[sensor] = values
    out["injection_interval"] = np.zeros(len(out), dtype='int8')
    out.loc[start_index:end_exclusive - 1, "injection_interval"] = 1
    out["synthetic_anomaly"] = np.zeros(len(out), dtype='int8')
    out.loc[start_index:end_exclusive - 1, "synthetic_anomaly"] = positive_phase.astype('int8')
    out["synthetic_event_id"] = pd.Series(pd.NA, index=out.index, dtype='string')
    out.loc[start_index:end_exclusive - 1, "synthetic_event_id"] = event_id
    out["anomaly_id"] = pd.Series(pd.NA, index=out.index, dtype='string')
    positive_indices = start_index + np.flatnonzero(positive_phase)
    out.loc[positive_indices, "anomaly_id"] = event_id

    metadata = {
        "experiment_id":event_id, "sensor":sensor, "fault_type":fault_type,
        "severity_sd":severity_sd, "duration":duration,
        "start_index":start_index, "end_index":end_exclusive - 1,
        "start_timestamp":str(clean_df.at[start_index, TIMESTAMP_COL]),
        "end_timestamp":str(clean_df.at[end_exclusive - 1, TIMESTAMP_COL]),
        "seed":int(seed), "direction":signed_direction if amplitude is not None else None,
        "n_labeled_readings":int(positive_phase.sum()),
        "n_changed_readings":int(changed_within.sum()),
        "is_negative_control":int(fault_type == "gaussian_noise"),
        "reference_source":reference_source,
        "benchmark_split":BENCHMARK_SPLIT,
        "benchmark_source_csv":str(BENCHMARK_PATH),
        "baseline_sd":sd,
        "benchmark_csv":"",
    }
    return out, metadata

## 5. Generate and export benchmark CSVs

One CSV is saved per experiment in `OUTPUT_DIR / "benchmarks"`. The manifest records split, source path, fault type, affected sensor, severity, duration, start/end indices and timestamps, seed, and output filename.

Each CSV preserves the original `PW_0.5h` label and adds `injection_interval`, `synthetic_anomaly`, `synthetic_event_id`, and `anomaly_id`. These synthetic labels are **evaluation ground truth**, not model-input features. The majority of each experiment's rows are identical to those of the selected source.

In [6]:
def choose_start(sensor, fault_type, duration, seed):
    starts = eligible_cache.get(duration)
    if starts is None:
        starts = eligible_starts(duration)
    if fault_type == "stuck_sensor":
        arr = clean_benchmark[sensor].to_numpy()
        starts = np.asarray([
            s for s in starts if not np.all(np.isclose(arr[s:s+duration], arr[s-1], atol=1e-12, rtol=0))
        ], dtype=np.int64)
    if len(starts) == 0:
        raise ValueError(f"No valid position for {sensor}: {fault_type}, duration={duration}")
    # Independent RNG stream for start selection, separate from fault values.
    rng = np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, int(seed), 991]))
    return int(rng.choice(starts))


# Lightweight checks during CSV generation: source rows and future-failure labels are unchanged.
manifest_records = []
for experiment_number, spec in enumerate(specifications, start=1):
    event_id = f"E{experiment_number:06d}"
    experiment_seed = RANDOM_SEED + experiment_number
    start = choose_start(spec["sensor"], spec["fault_type"], spec["duration"], experiment_seed)
    benchmark, metadata = inject_fault(
        clean_benchmark, spec["sensor"], spec["fault_type"], start,
        spec["duration"], spec["severity_sd"], experiment_seed, event_id,
    )
    assert benchmark[FAILURE_HORIZON_COL].equals(clean_benchmark[FAILURE_HORIZON_COL])
    assert benchmark[TIMESTAMP_COL].equals(clean_benchmark[TIMESTAMP_COL])
    output_file = OUTPUT_DIR / "benchmarks" / f"{event_id}.csv"
    benchmark.to_csv(output_file, index=False)
    metadata["replicate"] = spec["replicate"]
    metadata["benchmark_csv"] = str(output_file)
    manifest_records.append(metadata)
    if experiment_number % 100 == 0:
        print(f"Exported {experiment_number:,} / {len(specifications):,}")

manifest = pd.DataFrame(manifest_records)
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
print(f"Exported {len(manifest):,} CSVs to {OUTPUT_DIR / 'benchmarks'}")
print(f"Manifest: {OUTPUT_DIR / 'manifest.csv'}")
print(manifest[["experiment_id", "sensor", "fault_type", "severity_sd", "duration", "benchmark_csv"]].head(10).to_string(index=False))

Exported 60 CSVs to synthetic_validation_fault_outputs\benchmarks
Manifest: synthetic_validation_fault_outputs\manifest.csv
experiment_id           sensor        fault_type  severity_sd  duration                                             benchmark_csv
      E000001 CMS air pressure    gaussian_noise         0.05         5 synthetic_validation_fault_outputs\benchmarks\E000001.csv
      E000002 CMS air pressure    gaussian_noise         0.05         5 synthetic_validation_fault_outputs\benchmarks\E000002.csv
      E000003 CMS air pressure             spike         2.00         1 synthetic_validation_fault_outputs\benchmarks\E000003.csv
      E000004 CMS air pressure             spike         2.00         1 synthetic_validation_fault_outputs\benchmarks\E000004.csv
      E000005 CMS air pressure       short_burst         2.00         5 synthetic_validation_fault_outputs\benchmarks\E000005.csv
      E000006 CMS air pressure       short_burst         2.00         5 synthetic_validation_fau

## 6. Use the generated outputs

- **Development benchmarks:** `synthetic_validation_fault_outputs/benchmarks/E000001.csv`, etc. Each is an independently modified copy of **`val.csv`** containing one controlled event.
- **Development manifest:** `synthetic_validation_fault_outputs/manifest.csv` maps experiment IDs to their source, sensor, fault type, severity, duration, timestamps, random seed, and output path.
- **Final-evaluation benchmarks:** set `BENCHMARK_SPLIT = "test"` and rerun to write the **separate** `synthetic_test_fault_outputs/` folder. Only do this when model hyperparameters, reading windows, and threshold-selection methods are fixed.

**For the model developer:** fit Isolation Forest and preprocessing on the *original* training data. Use synthetic validation benchmarks to develop and calibrate the model; use synthetic test benchmarks only for the final held-out evaluation. Do not supply `PW_0.5h`, `synthetic_anomaly`, `injection_interval`, or event identifiers as training/prediction features.

**Interpretation:** `synthetic_anomaly = 1` marks an injected *sensor-level* fault. It is different from `PW_0.5h = 1`, which indicates a *future machine failure within 30 minutes*. The filter `PW_0.5h = 0` merely excludes labeled imminent-failure periods—it does not prove baseline health. Anomaly detection and future machine-failure prediction should be evaluated separately.

**Practical limitations:** every CSV is a full copy of the source with one event. Avoid concatenating the files as though all rows were independent training examples. A missing-value policy is needed for `dropout` when scoring with Isolation Forest. Full mode can create very large outputs; run pilot mode first.

**Reruns:** CSV files already in the output directory are not deleted automatically. If you switch between pilot and full modes in the same split, use a fresh output directory or move/archive the old benchmark files so obsolete CSVs are not mistaken for the current run. The manifest contains only experiments generated by the most recent run.
